# Genetic Algorithms

## 1. Local Beam Search

In [ ]:
class Action:
    pass

class State:
    def is_goal(self) -> bool:
        pass

    def successor_fn(self) -> list[tuple[Action, State]]:
        pass

    def eval(self) -> float:
        pass

In [8]:
import heapq

class BoundedPriorityQueue:
    def __init__(self, k: int):
        self.arr = []
        self.k = k

    def __len__(self):
        return len(self.arr)

    def empty(self) -> bool:
        return len(self) == 0

    def push(self, element: State, key: float) -> None:
        if len(self) < self.k:
            self.arr.append((key, element))
        elif self.arr[0][0] < key:
            heapq.heappop(self.arr)
            heapq.heappush(self.arr, (key, element))

    def drain(self) -> list[State]:
        arr = [element for _, element in self.arr]
        self.arr = []
        return arr

In [ ]:
def local_beam_search(states: list[State], iterations: int) -> State:
    k = len(states)
    for _ in range(0, iterations):
        candidates = BoundedPriorityQueue(k)
        for state in states:
            for action, next_state in state.successor_fn():
                if next_state.is_goal():
                    return next_state
                candidates.push(next_state, next_state.eval())
        states = candidates.drain()
    return states[0]

## 2. Genetic Algorithms

### 3.1 Definitions

**Genetic algorithms** = **stochastic local beam search** + **generate successors from pairs of states**

Genetic algorithms are:

- **metaheuristic** inspired by process of *natural selection* that belongs to the larger class of
evolutionary algorithms. Heuristics are problem-specific while metaheuristics provide generalized templates.
- commonly used in generation of high-quality solutions to optimization and search problems.
- a type of local search that mimics evolution by working on **populations of strings**,
which encode possible solutions and combines them based on a fitness function to produce individuals that are more fit

In [ ]:
# this is a pseudocode
def genetic_algorithm():
    population = population_initialization()
    scores = fitness_fn(population)
    while is_goal(population, scores):
        children = []
        while does_it_need_more_children(children):
            p1 = parent_selection(population, scores, None)
            p2 = parent_selection(population, scores, p1)
            child = crossover(p1, p2)
            child = mutation(child)
            children.append(child)
        population = population_generation(population, scores, children)
        scores = fitness_fn(population)
    return population, scores

The main steps are:

- initialization
- fitness function
- **selection**
- **crossover**
- **mutation**

### 2.2 Initialization. Population. Individual

Similar to local beam search, **GAs** (Genetic Algorithms) start with **k randomly generated states**, called the
**population**. Each state (also called **individual**) is represented as string over a finite alphabet.

### 2.3 Fitness Function

The **fitness function** is just GA terminology for evaluation function,
which returns higher values for better states. For example in N-queens problem, this can be the number of nonattacking pairs of queens.

## 3. Homework

### 3.1 Traveling Salesman Problem

Given a list of cities and the distances between each pair of cities,
what is the shortest possible route that visits each city exactly once?
It is **not** required to return to the starting position.

What will be the shape of an individual? - Permutation of [1, 2, 3, 4, 5, ..., n]

What will be the fitness function? - $ \sum_{i = 1}^{n} dist(p[i-1], p[i])$

### 3.2 Knapsack Problem

Given a set of items, each with a weight and a value,
determine which items to include in the collection
so that the total weight is less than or equal to a given limit
and the total value is as large as possible.

What will be the shape of an individual? - Boolean array
(boolean value for each item, true means included, false excluded).

What will be the fitness function? - if the total weight of the items is larger than the limit 0
else the total value of the items.


## 4. Selection

Selection is the stage of a genetic algorithm in which individual genomes are chosen from a population for
later breeding (using the crossover operator).

Selection methods:

- Roulette Wheel Selection (Fitness Proportionate Selection)
- Tournament Selection
- Rank-Based Selection
- Stochastic Universal Sampling (SUS)
- Elitism
- Truncation Selection

In [2]:
population = ["A", "B", "C", "D"]
scores = [60, 5, 30, 5]

### 4.1 Roulette Wheel Selection

In [6]:
import random

def roulette_wheel_selection(population: list[str], scores: list[float], n: int) -> list[str]:
    """
    Divides the interval [0, 1) to slices, proportionate to the scores.
    Scores must be non-negative.
    One very fit individual nukes the diversity.
    If all scores are almost the same, this selection behaves like random selection.
    """
    selected = []
    total_score = sum(scores)
    for _ in range(0, n):
        spin = random.uniform(0, total_score)
        accumulated_score = 0
        for individual, score in zip(population, scores):
            accumulated_score += score
            if accumulated_score >= spin:
                selected.append(individual)
                break
    return selected

In [8]:
print(roulette_wheel_selection(population, scores, 10))

['C', 'A', 'D', 'D', 'A', 'C', 'C', 'A', 'A', 'D']


### 4.2 Tournament Selection

In [11]:
import random

def tournament_selection(population: list[str], scores: list[float], tournament_size:int, n: int) -> list[str]:
    """Selects k unique participants, organizes tournament, the one with the highest score wins."""

    selected = []
    for _ in range(0, n):
        participants_index = random.sample(range(len(population)), tournament_size)
        champion_index = max(participants_index, key=lambda x: scores[x])
        selected.append(population[champion_index])
    return selected

In [12]:
print(tournament_selection(population, scores, 2, 10))

['D', 'A', 'A', 'C', 'B', 'B', 'C', 'B', 'A', 'A']


### 4.3 Rank-Based Selection

In [13]:
import random

def rank_based_selection(population: list[str], scores: list[float], n: int) -> list[str]:
    # sorting the indices by the fitness score
    sorted_indices = sorted(range(len(scores)), key=lambda x: scores[x])
    # it has the shape [(rank, index in population)]
    ranked_indices =  enumerate(sorted_indices, start=1)
    ranked_population = [(rank, population[population_index]) for rank, population_index in ranked_indices]
    # 1 + 2 + 3 + 4 + ... + n = n * (n+1) / 2
    total = len(population) * (len(population) + 1) // 2
    selected = []
    for _ in range(0, n):
        spin = random.uniform(0, total)
        accumulated = 0
        for rank, individual in ranked_population:
            accumulated += rank
            if accumulated >= spin:
                selected.append(individual)
                break
    return selected

In [14]:
rank_based_selection(population, scores, 10)

['C', 'A', 'C', 'A', 'A', 'B', 'B', 'C', 'D', 'A']

### 4.4 Elitism and Truncation Selection

Elitism - Get the best k individuals and put them straight
into the next generation without crossover or mutation

Truncation Selection - Receives fraction T (example 50%).
Only the individuals in top T fraction can be parents. Everyone below the cutoff line is discarded.
Parents are usually pick uniformly in random.

## 5. Crossover

Crossover is a genetic operator used to vary the programming of
a chromosome (or chromosomes) from one generation to the next.

It is analogous to reproduction and biological crossover, upon which GA are based.

Crossover is a process of taking more than one parent solutions and producing a child solution from them.

Crossover methods:

- One-Point Crossover
- Two-Point Crossover
- Partially-mapped Crossover
- Cycle Crossover
- Uniform Crossover

### 5.1 One-Point Crossover

In [45]:
def one_point_crossover(parent1: list[int], parent2:list[int] , m: int):
    genes_count = len(parent1)
    child1 = [None] * genes_count
    child2 = [None] * genes_count

    child1[:m] = parent1[:m]
    child2[:m] = parent2[:m]
    child1[m:] = parent2[m:]
    child2[m:] = parent1[m:]

    return child1, child2

In [46]:
parent1 = [1, 2, 3, 4, 5]
parent2 = [5, 4, 3, 2, 1]
m = 3
print(one_point_crossover(parent1, parent2, m))

([1, 2, 3, 2, 1], [5, 4, 3, 4, 5])


In [47]:
def one_point_permutation_crossover(parent1: list[int], parent2: list[int] , m: int):
    genes_count = len(parent1)
    child1 = [None] * genes_count
    child2 = [None] * genes_count

    child1[:m] = parent1[:m]
    child2[:m] = parent2[:m]
    child1_genes = set(child1[:m])
    child2_genes = set(child2[:m])

    parent2_ind = 0
    for i in range(m, genes_count):
        while parent2[parent2_ind] in child1_genes:
            parent2_ind += 1
        child1[i] = parent2[parent2_ind]
        child1_genes.add(child1[i])

    parent1_ind = 0
    for i in range(m, genes_count):
        while parent1[parent1_ind] in child2_genes:
            parent1_ind += 1
        child2[i] = parent1[parent1_ind]
        child2_genes.add(child2[i])

    return child1, child2

In [48]:
parent1 = [1, 2, 3, 4, 5]
parent2 = [5, 4, 3, 2, 1]
m = 3
print(one_point_permutation_crossover(parent1, parent2, m))

([1, 2, 3, 5, 4], [5, 4, 3, 1, 2])


### 5.2 Two-Point Crossover

In [49]:
import itertools

def two_point_crossover(parent1: list[int], parent2:list[int] , l: int, r:int):
    genes = len(parent1)
    child1 = [None] * genes
    child2 = [None] * genes

    child1[l:r] = parent1[l:r]
    child2[l:r] = parent2[l:r]
    child1[:l] = parent2[:l]
    child2[:l] = parent1[:l]
    child1[r:] = parent2[r:]
    child2[r:] = parent1[r:]

    return child1, child2

In [50]:
parent1 = [1, 2, 3, 4, 5, 6, 7]
parent2 = [7, 6, 2, 4, 3, 5, 1]
l = 3
r = 5
print(two_point_crossover(parent1, parent2, l, r))

([7, 6, 2, 4, 5, 5, 1], [1, 2, 3, 4, 3, 6, 7])


In [38]:
def two_point_permutation_crossover(parent1: list[int], parent2: list[int] , l: int, r: int):
    genes_count = len(parent1)
    child1 = [None] * genes_count
    child2 = [None] * genes_count


    child1[l:r] = parent1[l:r]
    child2[l:r] = parent2[l:r]
    child1_genes = set(child1[l:r])
    child2_genes = set(child2[l:r])

    parent2_ind = r
    for i in itertools.chain(range(r, genes_count), range(l)):
        while parent2[parent2_ind] in child1_genes:
            parent2_ind = (parent2_ind+1) % genes_count
        child1[i] = parent2[parent2_ind]
        child1_genes.add(child1[i])

    parent1_ind = r
    for i in itertools.chain(range(r, genes_count), range(l)):
        while parent1[parent1_ind] in child2_genes:
            parent1_ind = (parent1_ind+1) % genes_count
        child2[i] = parent1[parent1_ind]
        child2_genes.add(child2[i])

    return child1, child2

In [51]:
parent1 = [1, 2, 3, 4, 5, 6, 7]
parent2 = [7, 6, 2, 4, 3, 5, 1]
l = 2
r = 5
print(two_point_permutation_crossover(parent1, parent2, l, r))

([6, 2, 3, 4, 5, 1, 7], [1, 5, 2, 4, 3, 6, 7])


### 5.3 Partial-mapped Crossover

In [52]:
def partial_mapped_crossover(parent1: list[int], parent2: list[int] , l: int, r: int):
    genes_count = len(parent1)
    child1 = [None] * genes_count
    child2 = [None] * genes_count


    child1[l:r] = parent1[l:r]
    child2[l:r] = parent2[l:r]
    child1_segment = set(child1[l:r])
    child2_segment = set(child2[l:r])
    indices_in_parent1 = {gene: ind for ind, gene in enumerate(parent1)}
    indices_in_parent2 = {gene: ind for ind, gene in enumerate(parent2)}

    for i in itertools.chain(range(l), range(r, genes_count)):
        gene = parent2[i]
        while gene in child1_segment:
            gene = parent2[indices_in_parent1[gene]]
        child1[i] = gene

    for i in itertools.chain(range(l), range(r, genes_count)):
        gene = parent1[i]
        while gene in child2_segment:
            gene = parent1[indices_in_parent2[gene]]
        child2[i] = gene

    return child1, child2

In [53]:
parent1 = [1, 2, 3, 4, 5, 6, 7]
parent2 = [6, 3, 4, 1, 2, 7, 5]
l = 2
r = 5
print(partial_mapped_crossover(parent1, parent2, l, r))

([6, 1, 3, 4, 5, 7, 2], [3, 5, 4, 1, 2, 6, 7])


### 5.4 Cycle Crossover

In [56]:
def cycle_crossover(parent1: list[int], parent2: list[int]):
    genes_count = len(parent1)
    child1 = [None] * genes_count
    child2 = [None] * genes_count

    indices_in_parent1 = {gene: ind for ind, gene in enumerate(parent1)}
    indices_in_parent2 = {gene: ind for ind, gene in enumerate(parent2)}

    for i in range(genes_count):
        if child1[i] is not None:
            continue

        c = i
        while child1[c] is None:
            child1[c] = parent1[c]
            child2[c] = parent2[c]
            c = indices_in_parent1[parent2[c]]

        parent1, parent2 = parent2, parent1
        indices_in_parent1, indices_in_parent2 = indices_in_parent2, indices_in_parent2

    return child1, child2

In [57]:
parent1 = [8, 4, 7 ,3, 6, 2, 5, 1, 9, 0]
parent2 = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
print(cycle_crossover(parent1, parent2))

([8, 1, 2, 3, 4, 5, 6, 7, 9, 0], [0, 4, 7, 3, 6, 2, 5, 1, 8, 9])


### 5.5 Uniform crossover

In [58]:
import random

def uniform_crossover(parent1: list[int], parent2: list[int]):
    child = [parent1[i] if random.choice([False, True]) else parent2[i] for i in range(len(parent1))]
    return child

In [62]:
parent1 = [1, 2, 3, 4, 5, 6, 7, 8, 9, 1]
parent2 = [2, 3, 4, 5, 5, 7, 5, 9, 9, 0]
print(uniform_crossover(parent1, parent2))

[1, 2, 4, 5, 5, 7, 5, 8, 9, 0]


## 6. Mutation

Mutation is a genetic operator used to maintain
genetic diversity from one generation of a population
of genetic algorithm chromosomes to the next.

It is analogous to biological mutation.

![GA Mutation Methods](../../images/ga-mutation.svg)